# Analyzing FarmBurg's A/B Test — Practice Skeleton (Extended)

**Product context:** Brian (PM at FarmBurg) ran a three-arm A/B test on a microtransaction upgrade package. Groups were offered different prices:
- Group A → **$0.99**
- Group B → **$1.99**
- Group C → **$4.99**

Data file: `clicks.csv` (columns: `user_id`, `group`, `is_purchase`).

Your goals in this extended practice notebook:
1. Reproduce the original analysis (contingency → Chi-Square → binomial tests vs revenue targets).
2. Add missing pieces: visualizations, confidence intervals, actual revenue, effect-size style metrics.
3. Practice *alternate* implementations of the same statistical tests.
4. Run a **simulation playground** where you change the weekly revenue target, sample size, or assumed true rates and watch the recommendation change.

Work cell-by-cell. Fill every `# TODO` before moving on. The companion **Solution** notebook contains complete runnable code if you get stuck.


## Analysis Flowchart

This flowchart guides the full decision process for FarmBurg's microtransaction pricing experiment.

```mermaid
flowchart TD
    A[Load clicks.csv<br/>user_id, group, is_purchase] --> B[Explore data<br/>head, value counts, rates]
    B --> C[Build Contingency Table<br/>group × is_purchase]
    C --> D[Chi-Square Test of Independence<br/>Is purchase rate associated with group?]
    D --> E{Significant<br/>p < 0.05?}
    E -->|Yes| F[BUT: Wrong business question!<br/>We care about revenue target, not just which has higher rate]
    E -->|No| Z[Stop / Re-examine design]
    F --> G[Define business goal:<br/>≥ $1000 weekly revenue]
    G --> H[Compute target purchase rate<br/>for each price: 0.99 / 1.99 / 4.99]
    H --> I[Extract sample size & sales<br/>per group A/B/C]
    I --> J[One-sample Binomial Tests<br/>H0: p ≤ p_target vs Ha: p > p_target]
    J --> K{Any group significantly<br/>above its target rate?}
    K -->|Only C| L[Recommend charge $4.99]
    K -->|None / Multiple| M[Sensitivity / more data / other levers]
    L --> N[Visualize rates, expected revenue,<br/>CIs, actual vs target]
    N --> O[Extended Practice & Simulation<br/>Change target revenue, sample size,<br/>or true rates → re-run decision]
    O --> P[Audience-ready summary<br/>Exec headline + technical appendix]
    style F fill:#fff3cd,stroke:#856404
    style L fill:#d4edda,stroke:#155724
    style O fill:#e6f3ff,stroke:#0066cc
```

**Key insight:** A significant Chi-Square only tells us the groups differ. The business needs the *right* price that is likely to clear a revenue hurdle. That requires a one-sided binomial test against a *derived* target proportion for each price point.


## 0. Setup
Import the libraries you will need and load the data.


In [ ]:
# TODO: import pandas as pd, numpy as np
# TODO: from scipy.stats import chi2_contingency, binomtest
# TODO: import matplotlib.pyplot as plt and seaborn as sns (for later visuals)

# TODO: load clicks.csv into abdata and display the first 5 rows with .head()


## 1. Inspect the data & basic rates

Print shape, value counts for `group` and `is_purchase`, and the purchase rate by group.
Which group has the highest observed purchase rate? Does that match the cheapest price?


In [ ]:
# TODO: print shape, group.value_counts(), is_purchase.value_counts()
# TODO: compute and print purchase rate by group (hint: groupby + value_counts(normalize=True) or mean after mapping Yes->1)


## 2. Contingency table & Chi-Square test

Create the contingency table of `group` × `is_purchase` with `pd.crosstab`. Print it.
Then run `chi2_contingency` and extract the p-value. Is it significant at α = 0.05?

**Discussion prompt:** A significant Chi-Square tells us the purchase *rates differ* across price points. Why is that *not* the question Brian should use to pick a price?


In [ ]:
# TODO: Xtab = pd.crosstab(...)
# TODO: print(Xtab)
# TODO: chi2, pval, dof, expected = chi2_contingency(Xtab)
# TODO: print(pval)  and decide is_significant


## 3. Business goal → target purchase rates

Brian needs **at least $1 000 of revenue per week** to justify the feature.
The test ran for one week, so `num_visits = len(abdata)` is a typical weekly traffic volume.

For each price point calculate:
- `num_sales_needed_xxx = 1000 / price`
- `p_sales_needed_xxx = num_sales_needed_xxx / num_visits`

Print the three target proportions. They should *decrease* as price increases.


In [ ]:
# TODO: num_visits = ...
# TODO: num_sales_needed_099, p_sales_needed_099
# TODO: same for 1.99 and 4.99
# TODO: print all three p_sales_needed_...


## 4. Observed sample sizes and sales per group

Extract `samp_size_099 / sales_099`, and the same for groups B and C.
You can use boolean indexing or look at the contingency table you already printed.


In [ ]:
# TODO: samp_size_099 = (abdata.group == 'A').sum()
# TODO: sales_099 = ((abdata.group == 'A') & (abdata.is_purchase == 'Yes')).sum()
# TODO: repeat for B and C; print all six numbers


## 5. One-sample binomial tests (the correct tests)

For each group test whether the observed number of purchases is *significantly greater* than the number implied by the target rate:

```python
binomtest(k=sales, n=samp_size, p=p_target, alternative='greater').pvalue
```

(Modern SciPy uses `binomtest`; older code used `binom_test`.)

Print `pvalueA`, `pvalueB`, `pvalueC`. Which (if any) are < 0.05?


In [ ]:
# TODO: pvalueA = binomtest(...).pvalue
# TODO: same for B and C
# TODO: print the three p-values


## 6. Decision

Based on the three p-values and α = 0.05, which price point (if any) has a purchase rate significantly above the rate required to hit $1 000 / week?
Set `final_answer` to the recommended price string (e.g. `'4.99'`) and print it.


In [ ]:
# TODO: final_answer = '???'
# TODO: print(final_answer)


## 7. Extended analysis — visuals & revenue (practice)

Calculate the *actual* revenue observed in the test week for each group (`sales * price`).
Also compute the *expected* weekly revenue if the true rate equalled the target rate.
Create a bar chart of observed purchase rates with a horizontal line for each group's target rate.
Optionally add a second chart of expected vs observed revenue.


In [ ]:
# TODO: revenue_A = sales_099 * 0.99  (etc.)
# TODO: print observed revenues and target-implied revenues
# TODO: plot purchase rates (bar) + target lines; save or show


## 8. Alternate implementations (practice)

Re-compute the binomial p-value for group C **without** calling `binomtest`:
use the survival function of the binomial distribution (`scipy.stats.binom.sf`).
Also compute a Wilson score confidence interval for the observed purchase rate of group C
(you can use the formula or `statsmodels.stats.proportion.proportion_confint`).


In [ ]:
# TODO: from scipy.stats import binom
# TODO: p_alt = binom.sf(sales_499 - 1, n=samp_size_499, p=p_sales_needed_499)
# TODO: Wilson or statsmodels CI for group C rate


## 9. Simulation playground

Write a small function that, given a weekly revenue target, a list of prices, and assumed true purchase rates (or the observed rates), re-runs the whole decision pipeline and returns which prices clear the hurdle at α = 0.05.

Then experiment:
- What happens if the revenue target is lowered to $400?
- What if weekly traffic doubles?
- What if the true rate for group C is only 3 %?

Document your findings in a markdown cell after the simulation.


In [ ]:
# TODO: def evaluate_pricing(target_revenue=1000, traffic=None, true_rates=None, alpha=0.05): ...
# TODO: call it with a few different scenarios and print results


## 10. Key takeaways (write your own)

After finishing the exercises, write 3–5 bullet points that capture what you learned about matching the statistical test to the business question, the danger of stopping at a Chi-Square, and how simulation helps explore sensitivity.


In [ ]:
# (optional) any final exploratory code
